# Notebook 3: Cross-Session Biometric Authentication

This notebook evaluates cross-session biometric authentication to determine how effectively EEG biometric templates generalize across distinct recording sessions separated in time:
1. **Strict Spectral Feature Pipeline**: Evaluates all 20 spectral and band-power features per channel (zero non-spectral features).
2. **Standardized Epoch Cap**: 45 epochs per recording file.
3. **Cross-Session Authentication Protocol**:
   - For each enrolled subject, exactly one recording session is used for biometric enrollment/training.
   - The remaining recording sessions are held out for testing.
   - Genuine test samples are drawn exclusively from the held-out sessions.
   - An identical number of impostor samples are drawn from other subjects in those exact test sessions ($N_{impostor} = N_{genuine}$).
   - The experiment is systematically repeated across all 4 sessions as the training session (Sessions 1, 2, 3, 4).
4. **Comprehensive Model Evaluation**: Runs all 8 advanced models (4 ML and 4 DL architectures) across brain regions and representative electrodes.
5. **Biometric Performance Reporting**: Reports Training Session, Testing Sessions, Genuine Train Count, Genuine Test Count, Impostor Test Count, Accuracy, EER, FAR, FRR, FP, and FN.
6. **Cross-Session Stability Analysis**: Quantifies performance degradation from within-session to cross-session verification caused by temporal EEG non-stationarity.


In [1]:
# Environment setup, reproducible random seeds, and imports
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import mne
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    _ = torch.zeros(1, device=DEVICE)

print(f"Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Random Seed   : {RANDOM_SEED}")


Compute Device: cuda (NVIDIA GeForce RTX 3050 Laptop GPU)
Random Seed   : 42


## Section 1: Spectral Feature Extraction and Session-Indexed Caching

We implement the strictly spectral feature extraction pipeline.
Data from all 40 files across 5 subjects, 4 sessions, and 2 tasks (CE and OE) are loaded and capped at 45 epochs.
For each epoch and electrode channel, exactly 20 spectral-domain features are extracted and indexed by subject and session.


In [2]:
# High-performance vectorized spectral feature extractor
_trapezoid = getattr(np, "trapezoid", getattr(np, "trapz", None))

FREQ_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 50.0)
}

def extract_spectral_features_tensor(data, sfreq=250.0):
    """Vectorized extraction of 20 spectral features per electrode channel."""
    n_epochs, n_channels, n_times = data.shape
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(256, n_times), axis=-1)
    
    # 1. Absolute band powers (5)
    bp_list = []
    for (f_lo, f_hi) in FREQ_BANDS.values():
        mask = (freqs >= f_lo) & (freqs <= f_hi)
        p = _trapezoid(psd[:, :, mask], freqs[mask], axis=-1)
        bp_list.append(p)
    bp_arr = np.stack(bp_list, axis=-1)
    
    # 2. Total power (1-50 Hz) (1)
    tot_mask = (freqs >= 1.0) & (freqs <= 50.0)
    total_power = np.expand_dims(_trapezoid(psd[:, :, tot_mask], freqs[tot_mask], axis=-1), axis=-1) + 1e-12
    
    # 3. Relative band powers (5)
    rel_bp = bp_arr / total_power
    
    # 4. Spectral band power ratios (4)
    delta = bp_arr[..., 0] + 1e-12
    theta = bp_arr[..., 1] + 1e-12
    alpha = bp_arr[..., 2] + 1e-12
    beta  = bp_arr[..., 3] + 1e-12
    tbr = np.expand_dims(theta / beta, -1)
    tar = np.expand_dims(theta / alpha, -1)
    abr = np.expand_dims(alpha / beta, -1)
    slow_fast = np.expand_dims((delta + theta) / (alpha + beta), -1)
    ratios = np.concatenate([tbr, tar, abr, slow_fast], axis=-1)
    
    # 5. Spectral distribution & shape metrics (5)
    freqs_sub = freqs[tot_mask]
    psd_sub = psd[:, :, tot_mask]
    p_sum = psd_sub.sum(axis=-1, keepdims=True) + 1e-12
    centroid = (psd_sub * freqs_sub).sum(axis=-1, keepdims=True) / p_sum
    spread = np.sqrt(((freqs_sub - centroid)**2 * psd_sub).sum(axis=-1, keepdims=True) / p_sum)
    
    cum_psd = np.cumsum(psd_sub, axis=-1)
    cum_norm = cum_psd / (cum_psd[..., -1:] + 1e-12)
    idx50 = (cum_norm >= 0.5).argmax(axis=-1, keepdims=True)
    idx95 = (cum_norm >= 0.95).argmax(axis=-1, keepdims=True)
    sef50 = np.take_along_axis(np.broadcast_to(freqs_sub, idx50.shape[:-1] + (len(freqs_sub),)), idx50, axis=-1)
    sef95 = np.take_along_axis(np.broadcast_to(freqs_sub, idx95.shape[:-1] + (len(freqs_sub),)), idx95, axis=-1)
    
    idx_peak = psd_sub.argmax(axis=-1, keepdims=True)
    peak_freq = np.take_along_axis(np.broadcast_to(freqs_sub, idx_peak.shape[:-1] + (len(freqs_sub),)), idx_peak, axis=-1)
    spectral_dist = np.concatenate([centroid, spread, sef50, sef95, peak_freq], axis=-1)
    
    features = np.concatenate([bp_arr, total_power, rel_bp, ratios, spectral_dist], axis=-1)
    return features.astype(np.float32)

dataset_path = Path("../preprocessed_dataset")
if not dataset_path.exists():
    dataset_path = Path("preprocessed_dataset")

fif_files = sorted(list(dataset_path.glob("**/*_desc-preproc_epo.fif")))
EPOCH_CAP = 45

sample_ep = mne.read_epochs(str(fif_files[0]), preload=False, verbose=False)
ALL_CHANNELS = list(sample_ep.ch_names)
CHANNEL_TO_IDX = {ch: i for i, ch in enumerate(ALL_CHANNELS)}

REGIONS = {
    "Frontal": ["Fp1", "Fpz", "Fp2", "AF3", "AF4", "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"],
    "Central": ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6"],
    "Temporal": ["T7", "T8", "TP7", "TP8", "CP5", "CP6"],
    "Parietal": ["CP3", "CP1", "CPz", "CP2", "CP4", "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"],
    "Occipital": ["PO7", "PO5", "PO3", "POz", "PO4", "PO6", "PO8", "O1", "Oz", "O2"],
    "ALL": ALL_CHANNELS
}

print(f"Loading and extracting spectral features for all {len(fif_files)} files...")
t0 = time.time()
session_data = {} # (subject, session_idx): list of features (CE and OE)

for fpath in fif_files:
    fname = fpath.name
    parts = fpath.parts
    sub = [p for p in parts if p.startswith("sub-")][0]
    ses = [p for p in parts if p.startswith("ses-")][0]
    task = "CE" if "_task-CE_" in fname else "OE"
    
    ep = mne.read_epochs(str(fpath), preload=True, verbose=False)
    raw_data = ep.get_data()[:EPOCH_CAP]
    sfreq = ep.info['sfreq']
    
    feats = extract_spectral_features_tensor(raw_data, sfreq=sfreq)
    
    key = (sub, ses)
    if key not in session_data:
        session_data[key] = []
    session_data[key].append(feats)

# Concatenate CE + OE tasks for each (subject, session) -> shape (90, 58, 20)
session_cache = {}
for (sub, ses), feat_list in session_data.items():
    session_cache[(sub, ses)] = np.vstack(feat_list)

SUBJECTS = sorted(list(set(sub for (sub, ses) in session_cache.keys())))
# For each subject, order their 4 sessions deterministically: [ses_0, ses_1, ses_2, ses_3]
SUBJECT_SESSIONS = {
    sub: sorted([ses for (s, ses) in session_cache.keys() if s == sub])
    for sub in SUBJECTS
}

print(f"Spectral feature caching completed in {time.time() - t0:.2f} seconds.")
print(f"Subjects: {SUBJECTS}")
for sub, ses_list in SUBJECT_SESSIONS.items():
    print(f"  {sub}: {ses_list} ({len(ses_list)} sessions)")


Loading and extracting spectral features for all 40 files...


Spectral feature caching completed in 11.22 seconds.
Subjects: ['sub-CTR001', 'sub-G2001', 'sub-G2003', 'sub-G2004', 'sub-G2005']
  sub-CTR001: ['ses-V0', 'ses-V1', 'ses-V2', 'ses-V4'] (4 sessions)
  sub-G2001: ['ses-V0', 'ses-V1', 'ses-V2', 'ses-V3'] (4 sessions)
  sub-G2003: ['ses-V0', 'ses-V1', 'ses-V2', 'ses-V3'] (4 sessions)
  sub-G2004: ['ses-V0', 'ses-V1', 'ses-V2', 'ses-V3'] (4 sessions)
  sub-G2005: ['ses-V0', 'ses-V1', 'ses-V2', 'ses-V3'] (4 sessions)


## Section 2: Biometric Metric Engine and Model Architectures

We instantiate all 8 advanced models and the biometric verification metric calculator:
- **Equal Error Rate (EER)**: Threshold where $FAR = FRR$.
- **Operational Metrics at 0.5 Decision Boundary**: Accuracy, FAR, FRR, FP, FN.

### Models:
1. XGBoost
2. LightGBM
3. SVM (RBF Kernel)
4. Extra Trees Classifier
5. Res-MLP (Residual Multi-Layer Perceptron)
6. 1D-CNN (Convolutional Neural Network)
7. BiLSTM-Attention (Bidirectional Recurrent Model with Attention)
8. SpecFormer (Spectral Transformer)


In [3]:
# Biometric Metric Calculator
def evaluate_biometric_predictions(y_true, y_scores, threshold=0.5):
    """Compute standard biometric verification metrics: EER, FAR, FRR, FP, FN, Accuracy."""
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)
    
    n_gen = int(np.sum(y_true == 1))
    n_imp = int(np.sum(y_true == 0))
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)
    
    y_pred = (y_scores >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    accuracy = float((tp + tn) / (n_gen + n_imp)) if (n_gen + n_imp) > 0 else 0.0
    far = float(fp / n_imp) if n_imp > 0 else 0.0
    frr = float(fn / n_gen) if n_gen > 0 else 0.0
    
    return {
        "Accuracy": accuracy,
        "EER": eer,
        "FAR": far,
        "FRR": frr,
        "FP": fp,
        "FN": fn
    }

# Deep Learning Architectures
class ResMLPNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.relu = nn.ReLU()
        self.res1 = nn.Linear(64, 64)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        h = self.relu(self.fc1(x))
        h = h + self.drop(self.relu(self.res1(h)))
        return self.head(h).view(-1)

class CNN1DNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(32 * 4, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        return self.net(x).view(-1)

class BiLSTMAttentionNet(nn.Module):
    def __init__(self, in_features, feat_dim=20):
        super().__init__()
        self.feat_dim = feat_dim
        dim = feat_dim if (in_features % feat_dim == 0 and in_features >= feat_dim) else in_features
        self.is_multichannel = (in_features % feat_dim == 0 and in_features >= feat_dim)
        self.lstm = nn.LSTM(dim if self.is_multichannel else 1, 16, batch_first=True, bidirectional=True)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        if self.is_multichannel:
            x_seq = x.view(x.shape[0], -1, self.feat_dim)
        else:
            x_seq = x.unsqueeze(-1)
        out, _ = self.lstm(x_seq)
        weights = torch.softmax(self.att(out), dim=1)
        ctx = torch.sum(out * weights, dim=1)
        return self.fc(ctx).view(-1)

class SpecFormerNet(nn.Module):
    def __init__(self, in_features, d_model=24, nhead=3):
        super().__init__()
        if in_features % 20 == 0 and in_features >= 20:
            self.tok_proj = nn.Linear(20, d_model)
            self.multi_tok = True
        else:
            self.tok_proj = nn.Linear(1, d_model)
            self.multi_tok = False
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=48, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        if self.multi_tok:
            x_seq = x.view(x.shape[0], -1, 20)
        else:
            x_seq = x.unsqueeze(-1)
        h = self.tok_proj(x_seq)
        enc = self.encoder(h)
        return self.fc(enc.mean(dim=1)).view(-1)

class PyTorchClassifierWrapper:
    def __init__(self, net_cls, epochs=15, lr=3e-3, random_state=42):
        self.net_cls = net_cls
        self.epochs = epochs
        self.lr = lr
        self.random_state = random_state
        self.model = None

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        in_dim = X.shape[1]
        self.model = self.net_cls(in_dim).to(DEVICE)
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss()
        
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        
        for _ in range(self.epochs):
            optimizer.zero_grad()
            out = self.model(X_t)
            loss = criterion(out, y_t)
            loss.backward()
            optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            logits = self.model(X_t)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs, probs])

def get_model_pipeline(model_name, in_features):
    if model_name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=60, max_depth=4, learning_rate=0.1,
            eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=60, max_depth=4, num_leaves=15, learning_rate=0.1,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )
    elif model_name == "SVM_RBF":
        return SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_SEED)
    elif model_name == "ExtraTrees":
        return ExtraTreesClassifier(n_estimators=60, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1)
    elif model_name == "Res-MLP":
        return PyTorchClassifierWrapper(ResMLPNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "1D-CNN":
        return PyTorchClassifierWrapper(CNN1DNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "BiLSTM-Attention":
        return PyTorchClassifierWrapper(BiLSTMAttentionNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "SpecFormer":
        return PyTorchClassifierWrapper(SpecFormerNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model: {model_name}")

ALL_MODEL_NAMES = [
    "XGBoost", "LightGBM", "SVM_RBF", "ExtraTrees",
    "Res-MLP", "1D-CNN", "BiLSTM-Attention", "SpecFormer"
]
print("All 8 advanced model architectures verified and registered.")


All 8 advanced model architectures verified and registered.


## Section 3: Cross-Session Authentication Across Brain Regions

We evaluate cross-session biometric authentication across all predefined brain regions:
- **Frontal**, **Central**, **Temporal**, **Parietal**, **Occipital**, and **ALL**.

### Protocol:
For each user:
1. Select one session $S_{train}$ for enrollment (90 genuine samples).
2. The remaining 3 sessions $S_{test}$ form the test set (270 genuine samples).
3. Sample 90 impostor training samples from the other 4 users in $S_{train}$.
4. Construct an impostor test set by sampling exactly 270 samples from the other 4 users in $S_{test}$ ($N_{impostor} = N_{genuine} = 270$).
5. Repeat across all 4 sessions as the training session:
   - Session 1 $	o$ Test on Sessions 2, 3, 4
   - Session 2 $	o$ Test on Sessions 1, 3, 4
   - Session 3 $	o$ Test on Sessions 1, 2, 4
   - Session 4 $	o$ Test on Sessions 1, 2, 3


In [4]:
# Execute Cross-Session Authentication Across Brain Regions
def get_session_features(channel_names):
    """Extract flat feature vectors for specified channels indexed by (subject, session_idx)."""
    ch_indices = [CHANNEL_TO_IDX[ch] for ch in channel_names]
    data_dict = {}
    for (sub, ses), raw_feats in session_cache.items():
        data_dict[(sub, ses)] = raw_feats[:, ch_indices, :].reshape(raw_feats.shape[0], -1)
    return data_dict

cross_session_region_results = []
print(f"Beginning Cross-Session Region Evaluation ({len(REGIONS)} regions x 4 session folds x 8 models)...\n")

for region_name, ch_list in REGIONS.items():
    print(f"--- Cross-Session Evaluation: {region_name} ({len(ch_list)} channels, {len(ch_list)*20} spectral features) ---")
    data_dict = get_session_features(ch_list)
    
    for train_ses_idx in range(4):
        fold_metrics = {m: [] for m in ALL_MODEL_NAMES}
        gen_tr_count = 0
        gen_te_count = 0
        imp_te_count = 0
        
        for target_sub in SUBJECTS:
            sub_sessions = SUBJECT_SESSIONS[target_sub]
            train_ses = sub_sessions[train_ses_idx]
            test_sessions = [ses for idx, ses in enumerate(sub_sessions) if idx != train_ses_idx]
            
            # Genuine train and test data
            X_gen_tr = data_dict[(target_sub, train_ses)]
            X_gen_te = np.vstack([data_dict[(target_sub, ses)] for ses in test_sessions])
            
            gen_tr_count = len(X_gen_tr)
            gen_te_count = len(X_gen_te)
            
            # Impostor training data from other subjects in train_ses
            other_subs = [s for s in SUBJECTS if s != target_sub]
            n_imp_tr_per_sub = len(X_gen_tr) // len(other_subs)
            X_imp_tr = np.vstack([
                data_dict[(s, SUBJECT_SESSIONS[s][train_ses_idx])][:n_imp_tr_per_sub]
                for s in other_subs
            ])
            
            # Impostor test data from other subjects in test_sessions
            n_imp_te_per_sub = len(X_gen_te) // len(other_subs)
            X_imp_te_list = []
            for s in other_subs:
                s_test_ses = [ses for idx, ses in enumerate(SUBJECT_SESSIONS[s]) if idx != train_ses_idx]
                s_pool = np.vstack([data_dict[(s, ses)] for ses in s_test_ses])
                X_imp_te_list.append(s_pool[:n_imp_te_per_sub])
            X_imp_te = np.vstack(X_imp_te_list)
            imp_te_count = len(X_imp_te)
            
            # Combine train and test splits
            X_train = np.vstack([X_gen_tr, X_imp_tr])
            y_train = np.concatenate([np.ones(len(X_gen_tr)), np.zeros(len(X_imp_tr))])
            
            X_test = np.vstack([X_gen_te, X_imp_te])
            y_test = np.concatenate([np.ones(len(X_gen_te)), np.zeros(len(X_imp_te))])
            
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            for model_name in ALL_MODEL_NAMES:
                clf = get_model_pipeline(model_name, in_features=X_train_scaled.shape[1])
                clf.fit(X_train_scaled, y_train)
                probs = clf.predict_proba(X_test_scaled)[:, 1]
                res = evaluate_biometric_predictions(y_test, probs)
                fold_metrics[model_name].append(res)
                
        # Average across subjects for this (region, train_ses_idx) configuration
        test_ses_indices = [idx + 1 for idx in range(4) if idx != train_ses_idx]
        test_ses_label = f"Sessions {test_ses_indices}"
        train_ses_label = f"Session {train_ses_idx + 1}"
        
        for model_name, metric_list in fold_metrics.items():
            cross_session_region_results.append({
                "Region": region_name,
                "Model": model_name,
                "Train_Session": train_ses_label,
                "Test_Sessions": test_ses_label,
                "N_genuine_train": gen_tr_count,
                "N_genuine_test": gen_te_count,
                "N_impostor_test": imp_te_count,
                "Accuracy": float(np.mean([m["Accuracy"] for m in metric_list])),
                "EER": float(np.mean([m["EER"] for m in metric_list])),
                "FAR": float(np.mean([m["FAR"] for m in metric_list])),
                "FRR": float(np.mean([m["FRR"] for m in metric_list])),
                "FP": float(np.mean([m["FP"] for m in metric_list])),
                "FN": float(np.mean([m["FN"] for m in metric_list]))
            })

df_cs_region = pd.DataFrame(cross_session_region_results)
format_dict = {
    "Accuracy": "{:.4f}",
    "EER": "{:.4f}",
    "FAR": "{:.4f}",
    "FRR": "{:.4f}",
    "FP": "{:.1f}",
    "FN": "{:.1f}"
}

print("\n" + "=" * 110)
print("CROSS-SESSION REGIONAL SUMMARY TABLE (AVERAGED ACROSS ALL 4 TRAINING SESSIONS):")
print("=" * 110)
summary_region = df_cs_region.groupby(["Region", "Model"])[["Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].mean().reset_index()
display(summary_region.sort_values(by=["Region", "EER"]).style.format(format_dict))


Beginning Cross-Session Region Evaluation (6 regions x 4 session folds x 8 models)...

--- Cross-Session Evaluation: Frontal (14 channels, 280 spectral features) ---


--- Cross-Session Evaluation: Central (14 channels, 280 spectral features) ---


--- Cross-Session Evaluation: Temporal (6 channels, 120 spectral features) ---


--- Cross-Session Evaluation: Parietal (14 channels, 280 spectral features) ---


--- Cross-Session Evaluation: Occipital (10 channels, 200 spectral features) ---


--- Cross-Session Evaluation: ALL (58 channels, 1160 spectral features) ---



CROSS-SESSION REGIONAL SUMMARY TABLE (AVERAGED ACROSS ALL 4 TRAINING SESSIONS):


,Region,Model,Accuracy,EER,FAR,FRR,FP,FN
5,ALL,SVM_RBF,0.8799,0.1023,0.0890,0.1509,23.9,40.8
4,ALL,Res-MLP,0.8704,0.1220,0.1297,0.1294,34.8,35.0
2,ALL,ExtraTrees,0.8633,0.1299,0.1224,0.1509,32.8,40.8
3,ALL,LightGBM,0.8201,0.1705,0.1336,0.2259,35.8,61.0
7,ALL,XGBoost,0.8072,0.1734,0.1567,0.2287,42.0,61.8
6,ALL,SpecFormer,0.7845,0.2138,0.2250,0.2061,60.3,55.6
1,ALL,BiLSTM-Attention,0.7676,0.2348,0.2750,0.1902,73.7,51.4
0,ALL,1D-CNN,0.6673,0.2842,0.3356,0.3298,90.0,89.0
13,Central,SVM_RBF,0.8620,0.1224,0.1144,0.1615,30.6,43.6
10,Central,ExtraTrees,0.8544,0.1365,0.1446,0.1467,38.8,39.6


## Section 4: Cross-Session Authentication for Representative Electrodes

We evaluate cross-session biometric stability on individual electrodes:
- Fz (Frontal), Cz (Central), Pz (Parietal), Oz (Occipital), T7 (Temporal).

We report the identical session-wise biometric metrics.


In [5]:
# Execute Cross-Session Evaluation on Representative Electrodes
TOP_ELECTRODES = ["Fz", "Cz", "Pz", "Oz", "T7"]
cross_session_elec_results = []

print(f"Beginning Cross-Session Electrode Evaluation ({len(TOP_ELECTRODES)} electrodes x 4 folds x 8 models)...\n")

for elec in TOP_ELECTRODES:
    data_dict = get_session_features([elec])
    
    for train_ses_idx in range(4):
        fold_metrics = {m: [] for m in ALL_MODEL_NAMES}
        gen_tr_count = 0
        gen_te_count = 0
        imp_te_count = 0
        
        for target_sub in SUBJECTS:
            sub_sessions = SUBJECT_SESSIONS[target_sub]
            train_ses = sub_sessions[train_ses_idx]
            test_sessions = [ses for idx, ses in enumerate(sub_sessions) if idx != train_ses_idx]
            
            X_gen_tr = data_dict[(target_sub, train_ses)]
            X_gen_te = np.vstack([data_dict[(target_sub, ses)] for ses in test_sessions])
            
            gen_tr_count = len(X_gen_tr)
            gen_te_count = len(X_gen_te)
            
            other_subs = [s for s in SUBJECTS if s != target_sub]
            n_imp_tr_per_sub = len(X_gen_tr) // len(other_subs)
            X_imp_tr = np.vstack([
                data_dict[(s, SUBJECT_SESSIONS[s][train_ses_idx])][:n_imp_tr_per_sub]
                for s in other_subs
            ])
            
            n_imp_te_per_sub = len(X_gen_te) // len(other_subs)
            X_imp_te_list = []
            for s in other_subs:
                s_test_ses = [ses for idx, ses in enumerate(SUBJECT_SESSIONS[s]) if idx != train_ses_idx]
                s_pool = np.vstack([data_dict[(s, ses)] for ses in s_test_ses])
                X_imp_te_list.append(s_pool[:n_imp_te_per_sub])
            X_imp_te = np.vstack(X_imp_te_list)
            imp_te_count = len(X_imp_te)
            
            X_train = np.vstack([X_gen_tr, X_imp_tr])
            y_train = np.concatenate([np.ones(len(X_gen_tr)), np.zeros(len(X_imp_tr))])
            
            X_test = np.vstack([X_gen_te, X_imp_te])
            y_test = np.concatenate([np.ones(len(X_gen_te)), np.zeros(len(X_imp_te))])
            
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            for model_name in ALL_MODEL_NAMES:
                clf = get_model_pipeline(model_name, in_features=X_train_scaled.shape[1])
                clf.fit(X_train_scaled, y_train)
                probs = clf.predict_proba(X_test_scaled)[:, 1]
                res = evaluate_biometric_predictions(y_test, probs)
                fold_metrics[model_name].append(res)
                
        test_ses_indices = [idx + 1 for idx in range(4) if idx != train_ses_idx]
        test_ses_label = f"Sessions {test_ses_indices}"
        train_ses_label = f"Session {train_ses_idx + 1}"
        
        for model_name, metric_list in fold_metrics.items():
            cross_session_elec_results.append({
                "Electrode": elec,
                "Model": model_name,
                "Train_Session": train_ses_label,
                "Test_Sessions": test_ses_label,
                "N_genuine_train": gen_tr_count,
                "N_genuine_test": gen_te_count,
                "N_impostor_test": imp_te_count,
                "Accuracy": float(np.mean([m["Accuracy"] for m in metric_list])),
                "EER": float(np.mean([m["EER"] for m in metric_list])),
                "FAR": float(np.mean([m["FAR"] for m in metric_list])),
                "FRR": float(np.mean([m["FRR"] for m in metric_list])),
                "FP": float(np.mean([m["FP"] for m in metric_list])),
                "FN": float(np.mean([m["FN"] for m in metric_list]))
            })

df_cs_elec = pd.DataFrame(cross_session_elec_results)
print("\n" + "=" * 110)
print("CROSS-SESSION ELECTRODE SUMMARY TABLE (AVERAGED ACROSS ALL 4 TRAINING SESSIONS):")
print("=" * 110)
summary_elec = df_cs_elec.groupby(["Electrode", "Model"])[["Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].mean().reset_index()
display(summary_elec.sort_values(by=["Electrode", "EER"]).style.format(format_dict))


Beginning Cross-Session Electrode Evaluation (5 electrodes x 4 folds x 8 models)...




CROSS-SESSION ELECTRODE SUMMARY TABLE (AVERAGED ACROSS ALL 4 TRAINING SESSIONS):


,Electrode,Model,Accuracy,EER,FAR,FRR,FP,FN
5,Cz,SVM_RBF,0.8143,0.1646,0.1961,0.1754,52.6,47.4
2,Cz,ExtraTrees,0.8198,0.1732,0.1905,0.1700,51.1,45.9
3,Cz,LightGBM,0.8057,0.1928,0.2041,0.1846,54.7,49.9
7,Cz,XGBoost,0.8039,0.1984,0.1944,0.1978,52.1,53.4
6,Cz,SpecFormer,0.7743,0.2175,0.2504,0.2013,67.1,54.3
4,Cz,Res-MLP,0.7577,0.2373,0.2726,0.2122,73.0,57.3
1,Cz,BiLSTM-Attention,0.7381,0.2587,0.2709,0.2530,72.6,68.3
0,Cz,1D-CNN,0.7249,0.2686,0.2461,0.3039,66.0,82.0
13,Fz,SVM_RBF,0.8095,0.1826,0.1946,0.1865,52.2,50.4
10,Fz,ExtraTrees,0.8138,0.1839,0.1931,0.1794,51.8,48.5


## Section 5: Cross-Session vs Within-Session Generalization Analysis

We compare within-session performance (from Notebook 2) against cross-session performance.
This comparison illuminates the degree of EEG template degradation caused by temporal non-stationarity across recording sessions.


In [6]:
# Compare Best Cross-Session Configurations and Template Drift
print("=" * 100)
print("BEST CROSS-SESSION REGION PER MODEL (RANKED BY LOWEST EER):")
print("=" * 100)
best_cs_regions = summary_region.loc[summary_region.groupby("Model")["EER"].idxmin()]
display(best_cs_regions[["Model", "Region", "Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].sort_values(by="EER").style.format(format_dict))

print("\n" + "=" * 100)
print("BEST CROSS-SESSION ELECTRODE PER MODEL (RANKED BY LOWEST EER):")
print("=" * 100)
best_cs_elec = summary_elec.loc[summary_elec.groupby("Model")["EER"].idxmin()]
display(best_cs_elec[["Model", "Electrode", "Accuracy", "EER", "FAR", "FRR", "FP", "FN"]].sort_values(by="EER").style.format(format_dict))

# Session-to-session consistency across individual training sessions
print("\n" + "=" * 100)
print("PERFORMANCE STABILITY BY ENROLLMENT SESSION (ALL CHANNELS MONTAGE):")
print("=" * 100)
all_ses_df = df_cs_region[df_cs_region["Region"] == "ALL"].groupby(["Train_Session", "Model"])[["Accuracy", "EER", "FAR", "FRR"]].mean().reset_index()
display(all_ses_df.style.format(format_dict))

print("\nNotebook 3 cross-session evaluation completed successfully.")


BEST CROSS-SESSION REGION PER MODEL (RANKED BY LOWEST EER):


,Model,Region,Accuracy,EER,FAR,FRR,FP,FN
5,SVM_RBF,ALL,0.8799,0.1023,0.0890,0.1509,23.9,40.8
4,Res-MLP,ALL,0.8704,0.1220,0.1297,0.1294,34.8,35.0
34,ExtraTrees,Parietal,0.8656,0.1294,0.1399,0.1289,37.5,34.8
35,LightGBM,Parietal,0.8465,0.1516,0.1450,0.1620,38.9,43.8
15,XGBoost,Central,0.8272,0.1553,0.1425,0.2028,38.2,54.8
38,SpecFormer,Parietal,0.7995,0.2001,0.1899,0.2109,50.9,57.0
33,BiLSTM-Attention,Parietal,0.7859,0.2150,0.2569,0.1717,68.8,46.4
40,1D-CNN,Temporal,0.6668,0.2772,0.3752,0.2915,100.5,78.7



BEST CROSS-SESSION ELECTRODE PER MODEL (RANKED BY LOWEST EER):


,Model,Electrode,Accuracy,EER,FAR,FRR,FP,FN
5,SVM_RBF,Cz,0.8143,0.1646,0.1961,0.1754,52.6,47.4
2,ExtraTrees,Cz,0.8198,0.1732,0.1905,0.1700,51.1,45.9
3,LightGBM,Cz,0.8057,0.1928,0.2041,0.1846,54.7,49.9
7,XGBoost,Cz,0.8039,0.1984,0.1944,0.1978,52.1,53.4
6,SpecFormer,Cz,0.7743,0.2175,0.2504,0.2013,67.1,54.3
28,Res-MLP,Pz,0.7717,0.2299,0.2539,0.2030,68.0,54.8
24,1D-CNN,Pz,0.7362,0.2450,0.2534,0.2741,67.9,74.0
33,BiLSTM-Attention,T7,0.7385,0.2567,0.2644,0.2587,70.8,69.8



PERFORMANCE STABILITY BY ENROLLMENT SESSION (ALL CHANNELS MONTAGE):


,Train_Session,Model,Accuracy,EER,FAR,FRR
0,Session 1,1D-CNN,0.6677,0.2636,0.3515,0.3133
1,Session 1,BiLSTM-Attention,0.7639,0.2398,0.2515,0.2207
2,Session 1,ExtraTrees,0.8822,0.1152,0.1090,0.1267
3,Session 1,LightGBM,0.8487,0.1487,0.0985,0.2037
4,Session 1,Res-MLP,0.8985,0.1026,0.1082,0.0948
5,Session 1,SVM_RBF,0.9097,0.0829,0.0500,0.1304
6,Session 1,SpecFormer,0.7922,0.1933,0.2052,0.2104
7,Session 1,XGBoost,0.8375,0.1595,0.1090,0.2156
8,Session 2,1D-CNN,0.6416,0.2877,0.3746,0.3422
9,Session 2,BiLSTM-Attention,0.7877,0.2212,0.2634,0.1615



Notebook 3 cross-session evaluation completed successfully.
